In [2]:
pip install ipynb

Note: you may need to restart the kernel to use updated packages.


In [ ]:
# job_browsing.ipynb
import tkinter as tk
from tkinter import messagebox, ttk
from styles import *
from ipynb.fs.full.data_manager import get_jobs, save_application, get_applications

class EmployeeDashboard(tk.Frame):

    def __init__(self, master, current_user):
        # Inherit as a responsive container frame nested inside the master window shell
        super().__init__(master)
        self.current_user = current_user
        self.configure(bg=BACKGROUND_COLOR)

        # Top Navigation Welcome Bar
        self.top_bar = tk.Frame(self, bg=PRIMARY_COLOR, height=60)
        self.top_bar.pack(fill="x", side="top")
        self.top_bar.pack_propagate(False)

        welcome_lbl = tk.Label(
            self.top_bar,
            text=f"Welcome back, {self.current_user['name']}!",
            font=FONT_SUBTITLE,
            fg="white",
            bg=PRIMARY_COLOR,
        )
        welcome_lbl.pack(side="left", padx=20)

        # Main Workspace Canvas Layout Frame Container
        self.main_container = tk.Frame(self, bg=BACKGROUND_COLOR)
        self.main_container.pack(fill="both", expand=True, padx=20, pady=20)

        # Automatically deploy home menu layout view panel
        self.show_dashboard_home()

    def clear_container(self):
        """Erases localized UI panel child elements cleanly prior to view switching."""
        for widget in self.main_container.winfo_children():
            widget.destroy()

    def show_dashboard_home(self):
        """Renders primary menu hub panels for job seekers."""
        self.clear_container()

        self.main_container.columnconfigure(0, weight=1)
        self.main_container.columnconfigure(1, weight=1)

        title_lbl = tk.Label(
            self.main_container,
            text="Job Seeker Portal Terminal",
            font=FONT_TITLE,
            bg=BACKGROUND_COLOR,
            fg=PRIMARY_COLOR,
        )
        title_lbl.grid(row=0, column=0, columnspan=2, pady=(0, 30), sticky="w")

        # Action Button Panel 1: Live Job Browser
        browse_btn = tk.Button(
            self.main_container,
            text="🔍 Find Available Positions\nBrowse real-time active vacancies",
            **BUTTON_STYLE,
            command=self.show_browse_jobs,
        )
        browse_btn.grid(row=1, column=0, padx=20, pady=10, ipady=15, sticky="nsew")

        # Action Button Panel 2: Historic Tracking Log Ledger
        history_btn = tk.Button(
            self.main_container,
            text="📁 Track Submitted Applications\nReview historical pipeline status logs",
            **BUTTON_STYLE,
            bg="#2ecc71",
            activebackground="#27ae60",
            command=self.show_applied_history,
        )
        history_btn.grid(row=1, column=1, padx=20, pady=10, ipady=15, sticky="nsew")

    def show_browse_jobs(self):
        """Displays searchable database table columns populated cleanly via live system logic handles."""
        self.clear_container()

        nav_frame = tk.Frame(self.main_container, bg=BACKGROUND_COLOR)
        nav_frame.pack(fill="x", pady=(0, 15))

        back_btn = tk.Button(
            nav_frame,
            text="← Back to Hub Dashboard",
            **BUTTON_STYLE,
            bg="#7f8c8d",
            activebackground="#2c3e50",
            command=self.show_dashboard_home,
        )
        back_btn.pack(side="left")

        # Context-Aware Filter Switch Toggle Checkbox
        self.filter_var = tk.BooleanVar(value=False)
        filter_chk = tk.Checkbutton(
            nav_frame,
            text="Filter: 'Entry-Level / No Experience' Only",
            variable=self.filter_var,
            command=self.load_jobs_into_table,
            font=FONT_BODY,
            bg=BACKGROUND_COLOR,
            fg=TEXT_COLOR,
            activebackground=BACKGROUND_COLOR,
        )
        filter_chk.pack(side="right", padx=10)

        # --- Treeview Component Render Block ---
        table_frame = tk.Frame(self.main_container, bg=BACKGROUND_COLOR)
        table_frame.pack(fill="both", expand=True)

        tree_scroll = tk.Scrollbar(table_frame)
        tree_scroll.pack(side="right", fill="y")

        columns = ("id", \"title\", \"skills\", \"experience\", \"description\")
        self.job_table = ttk.Treeview(
            table_frame,
            columns=columns,
            show="headings",
            yscrollcommand=tree_scroll.set,
        )
        tree_scroll.config(command=self.job_table.yview)

        self.job_table.heading("id", text="Job ID")
        self.job_table.heading("title", text="Job Designation Title")
        self.job_table.heading("skills", text="Core Skill Prerequisite")
        self.job_table.heading("experience", text="Required Experience Tier")
        self.job_table.heading("description", text="Role Description Overview")

        self.job_table.column("id", width=60, anchor=\"center\")
        self.job_table.column("title", width=180, anchor=\"w\")
        self.job_table.column("skills", width=160, anchor=\"w\")
        self.job_table.column("experience", width=140, anchor=\"center\")
        self.job_table.column("description", width=260, anchor=\"w\")

        self.job_table.pack(fill="both", expand=True)

        # Bottom Action Control Deck
        action_bar = tk.Frame(self.main_container, bg=BACKGROUND_COLOR)
        action_bar.pack(fill="x", pady=15)

        apply_btn = tk.Button(
            action_bar,
            text="🚀 Forward Application Profile to Employer",
            **BUTTON_STYLE,
            bg="#e67e22",
            activebackground="#d35400",
            command=self.submit_application,
        )
        apply_btn.pack(side="right")

        self.load_jobs_into_table()

    def load_jobs_into_table(self):
        """Pulls database records from data_manager schema without hardcoded fallbacks."""
        for item in self.job_table.get_children():
            self.job_table.delete(item)

        all_jobs = get_jobs()

        for job in all_jobs:
            if self.filter_var.get():
                exp_lower = job.get("experience", "").lower()
                if "no experience" not in exp_lower and "entry" not in exp_lower:
                    continue  

            self.job_table.insert(
                "",
                "end",
                values=(
                    job.get("id"),
                    job.get("title"),
                    job.get("skills"),
                    job.get("experience"),
                    job.get("description"),
                ),
            )

    def submit_application(self):
        """Extracts unique highlighted data references from row selection indexes safely."""
        selected_item = self.job_table.selection()

        if not selected_item:
            messagebox.showwarning("Missing Highlight Node", "Please select a target job opening row entry from the ledger framework first.")
            return

        row_values = self.job_table.item(selected_item, "values")
        selected_job_id = row_values[0]
        selected_title = row_values[1]

        success = save_application(selected_job_id, self.current_user["id"])

        if success:
            messagebox.showinfo("Submission Logged", f"Application context processing node for '{selected_title}' committed onto the server matrix successfully!")
        else:
            messagebox.showerror("Validation Conflict Exception", f"Policy Exception: Dual application logs are prohibited for position: '{selected_title}'.")

    def show_applied_history(self):
        """Retrieves and maps historical validation references tied to current user tracking ID."""
        self.clear_container()

        nav_frame = tk.Frame(self.main_container, bg=BACKGROUND_COLOR)
        nav_frame.pack(fill="x", pady=(0, 15))

        back_btn = tk.Button(
            nav_frame,
            text="← Back to Hub Dashboard",
            **BUTTON_STYLE,
            bg="#7f8c8d",
            activebackground="#2c3e50",
            command=self.show_dashboard_home,
        )
        back_btn.pack(side="left")

        lbl = tk.Label(
            self.main_container,
            text="Your Historical Application Submission Footprint Log",
            font=FONT_SUBTITLE,
            bg=BACKGROUND_COLOR,
            fg=TEXT_COLOR,
        )
        lbl.pack(pady=10, anchor="w")

        history_table = ttk.Treeview(self.main_container, columns=("job_id"), show="headings")
        history_table.heading("job_id", text="Successfully Appended Application Job ID References")
        history_table.pack(fill="both", expand=True)

        user_apps = get_applications(self.current_user["id"])
        for app in user_apps:
            history_table.insert("", "end", values=(app.get("job_id"),))